In [ ]:
!git clone https://github.com/EgoisticCoder/Project_Infra.git /kaggle/working/Project_Infra
%cd /kaggle/working/Project_Infra

In [ ]:

!pip install -q -U unsloth unsloth_zoo transformers trl peft datasets bitsandbytes accelerate gradio requests beautifulsoup4 pillow
!pip uninstall -y torchaudio

In [ ]:

from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

In [ ]:

  from pathlib import Path
  import json

  DATA_PATH = Path(
      "/kaggle/working/Project_Infra/ui-ux-model/"
      "dataset_gen_v2/v2_training_candidate.jsonl"
  )

  print(DATA_PATH.exists())
  print("Records:", sum(1 for _ in DATA_PATH.open()))
  

In [ ]:
# Kaggle: turn Internet on before running this cell.
!pip install -q -U unsloth unsloth_zoo transformers trl peft datasets bitsandbytes
# The notebook does not use audio. Kaggle can ship torchaudio built for a
# different CUDA version than PyTorch; transformers probes it on import.
!pip uninstall -y torchaudio

In [15]:
from pathlib import Path
import os

  # Exact path inside the cloned Project_Infra repository
DATA_PATH = Path(
      "/kaggle/working/Project_Infra/"
      "ui-ux-model/dataset_gen_v2/v2_training_candidate.jsonl"
  )

MODEL_NAME = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit"
OUTPUT_DIR = "/kaggle/working/forma_v1_lora_run1"

MAX_SEQ_LENGTH = 3072
CUSTOM_REPEAT = 1
MAX_STEPS = 10
USE_EXTERNAL_DATASETS = False
SEED = 3407

print("Dataset path:", DATA_PATH)
print("Exists:", DATA_PATH.exists())

if DATA_PATH.exists():
      print("Records:", sum(1 for _ in DATA_PATH.open(encoding="utf-8")))
else:
      print("Dataset not found. Searching...")
      matches = list(Path("/kaggle/working").rglob("v2_training_candidate.jsonl"))
      print(matches)

Dataset path: /kaggle/working/Project_Infra/ui-ux-model/dataset_gen_v2/v2_training_candidate.jsonl
Exists: True
Records: 348


In [17]:
import base64, hashlib, json, random
from io import BytesIO
from PIL import Image
from datasets import Dataset

random.seed(SEED)

def as_text(value):
    if value is None:
        return ''
    if isinstance(value, str):
        return value.strip()
    return json.dumps(value, ensure_ascii=False, indent=2)

def decode_image(value):
    if not value:
        return None
    if isinstance(value, Image.Image):
        return value.convert('RGB')
    if isinstance(value, dict):
        value = value.get('data') or value.get('bytes') or value.get('image')
    if isinstance(value, (bytes, bytearray)):
        try:
            return Image.open(BytesIO(value)).convert('RGB')
        except Exception:
            return None
    if not isinstance(value, str):
        return None
    try:
        if len(value) < 500 and Path(value).exists():
            return Image.open(value).convert('RGB')
        raw = value.split(',', 1)[1] if value.startswith('data:') else value
        return Image.open(BytesIO(base64.b64decode(raw))).convert('RGB')
    except Exception:
        return None

def normalise_record(row):
    # Already-normalized conversational examples.
    if isinstance(row.get('messages'), list):
        messages = row['messages']
        return {'messages': messages, 'has_image': any(
            isinstance(m.get('content'), list) and any(c.get('type') == 'image' for c in m['content'] if isinstance(c, dict))
            for m in messages if isinstance(m, dict)
        )}

    if row.get('prompt') and row.get('response') is not None:
        user_text = as_text(row['prompt'])
        answer = as_text(row['response'])
    elif row.get('status') == 'done' and row.get('final'):
        app = row.get('app_type', 'digital product')
        style = row.get('style', 'appropriate')
        user_text = f'Design and explain a high-quality UI/UX for a {app} in a {style} style. Include layout, interaction, responsive behavior, accessibility, design tokens, states, and implementation-ready details.'
        answer = row['final']
    else:
        return None
    if len(user_text) < 8 or len(answer) < 20:
        return None
    image = decode_image(row.get('screenshot_b64') or row.get('image'))
    content = [{'type': 'text', 'text': user_text}]
    if image is not None:
        content.insert(0, {'type': 'image', 'image': image})
    return {'messages': [
        {'role': 'user', 'content': content},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': answer}]},
    ], 'has_image': image is not None}

rows, rejected = [], 0
with DATA_PATH.open(encoding='utf-8') as f:
    for line in f:
        try:
            item = normalise_record(json.loads(line))
        except Exception:
            item = None
        if item is None:
            rejected += 1
        else:
            rows.append(item)

# Exact duplicate conversations waste capacity and can bias the adapter.
unique, seen = [], set()
for item in rows:
    fingerprint = hashlib.sha256(json.dumps(item['messages'], ensure_ascii=False, default=str).encode()).hexdigest()
    if fingerprint not in seen:
        seen.add(fingerprint); unique.append(item)
deduped = len(rows) - len(unique)
rows = unique
custom_rows = rows
print({'raw_rows': sum(1 for _ in DATA_PATH.open(encoding='utf-8')), 'usable_custom_rows': len(custom_rows), 'rejected': rejected, 'duplicates_removed': deduped})

{'raw_rows': 348, 'usable_custom_rows': 0, 'rejected': 348, 'duplicates_removed': 0}


In [18]:
# Download capped, task-relevant external data. Kaggle Internet must be enabled.
from datasets import load_dataset

def first_present(row, *keys):
    for key in keys:
        if key in row and row[key] is not None:
            return row[key]
    return None

def external_example(image, prompt, answer):
    image = decode_image(image)
    if image is None or len(as_text(answer)) < 20:
        return None
    image = image.convert('RGB')
    image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.Resampling.LANCZOS)
    return {'messages': [{'role': 'user', 'content': [
        {'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}
    ]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': as_text(answer)}]}], 'has_image': True}

websight_rows, rico_rows = [], []
if USE_EXTERNAL_DATASETS:
    # WebSight: screenshot -> HTML/CSS. The conservative cap keeps decoded images
    # within Kaggle RAM while still providing a useful code prior.
    web_stream = load_dataset('HuggingFaceM4/WebSight', 'v0.2', split='train', streaming=True)
    for item in web_stream: 
        image = first_present(item, 'image', 'images')
        html = first_present(item, 'text', 'html', 'code')
        ex = external_example(image, 'Recreate this webpage as a responsive, accessible HTML/CSS/SVG implementation. Return only the implementation code.', html)
        if ex is not None:
            websight_rows.append(ex)
        if len(websight_rows) >= WEBSIGHT_LIMIT:
            break

    # Rico: screenshot + view hierarchy -> structured visual grounding. We turn the
    # annotations into an auxiliary answer format instead of mixing raw detection rows
    # into website-code answers.
    rico_stream = load_dataset('Voxel51/rico', split='train', streaming=True)
    for item in rico_stream:
        image = first_present(item, 'image', 'img', 'filepath')
        detections = item.get('detections', {})
        if isinstance(detections, dict):
            detections = detections.get('detections', [])
        if not isinstance(detections, list) or not detections:
            continue
        elements = []
        for det in detections[:80]:
            if not isinstance(det, dict):
                continue
            box = det.get('bounding_box', det.get('bbox'))
            label = det.get('label') or det.get('type') or det.get('content_or_function') or 'ui_element'
            if box is not None:
                elements.append({'label': label, 'bbox': box, 'clickable': det.get('clickable')})
        if not elements:
            continue
        answer = json.dumps({'task': 'UI element grounding', 'elements': elements}, ensure_ascii=False)
        ex = external_example(image, 'Inspect this mobile UI screenshot and return the visible UI elements with their labels, bounding boxes, and clickability. Preserve the supplied JSON schema.', answer)
        if ex is not None:
            rico_rows.append(ex)
        if len(rico_rows) >= RICO_LIMIT:
            break

# Oversample project-specific behavior, then add the capped auxiliary data.
rows = custom_rows * CUSTOM_REPEAT + websight_rows + rico_rows
random.shuffle(rows)
split = max(1, int(len(rows) * 0.10))
eval_rows, train_rows = rows[:split], rows[split:]
train_ds, eval_ds = Dataset.from_list(train_rows), Dataset.from_list(eval_rows)
has_images = any(x['has_image'] for x in rows)
print({'custom': len(custom_rows), 'custom_after_repeat': len(custom_rows) * CUSTOM_REPEAT, 'websight': len(websight_rows), 'rico': len(rico_rows), 'train': len(train_ds), 'validation': len(eval_ds), 'has_images': has_images})

{'custom': 0, 'custom_after_repeat': 0, 'websight': 0, 'rico': 0, 'train': 0, 'validation': 0, 'has_images': False}


In [ ]:
pip install --upgrade --force-reinstall Pillow

In [19]:
import torch
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
    max_seq_length=MAX_SEQ_LENGTH,
)

model = FastVisionModel.get_peft_model(
    model,
    # finetune_vision_layers=has_images,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
print('Vision adapters enabled:')

==((====))==  Unsloth 2026.9.7: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 3.958 GiB
no_split classes   : ['Qwen3VLTextDecoderLayer', 'Qwen3VLVisionBlock']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 8.170 GiB requested
tied to head       : ['model.language_model.embed_tokens']
  cuda:0  budget  10.24 GiB  weights  2.037 GiB  free  8.203 GiB  reserve  8.170 GiB
  cuda:1  budget  10.42 GiB  weights  1.920 GiB  free  8.495 GiB  reserve  8.078 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 11.378 -> 10.240

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Vision adapters enabled:


In [20]:

  from pathlib import Path
  import json
  import hashlib
  from datasets import Dataset

  DATA_PATH = Path(
      "/kaggle/working/Project_Infra/"
      "ui-ux-model/dataset_gen_v2/v2_training_candidate.jsonl"
  )

  assert DATA_PATH.exists(), f"Missing dataset: {DATA_PATH}"

  def text(value):
      if value is None:
          return ""
      if isinstance(value, str):
          return value
      return json.dumps(value, ensure_ascii=False, indent=2)

  rows = []
  seen = set()

  with DATA_PATH.open(encoding="utf-8") as file:
      for line in file:
          if not line.strip():
              continue

          row = json.loads(line)

          if not row.get("task") or not row.get("corrected_code"):
              continue

          user_prompt = f"""
  Review and improve this UI/UX design.

  Task:
  {text(row.get("task"))}

  Constraints:
  {text(row.get("constraints"))}

  Research evidence:
  {text(row.get("research_evidence"))}

  Design specification:
  {text(row.get("design_spec"))}

  Critic feedback:
  {text(row.get("critic_feedback"))}

  Return specific UI/UX recommendations followed by a complete corrected HTML/CSS implementation.
  """.strip()

          assistant_answer = f"""
  Recommendations:
  {text(row.get("critic_feedback"))}

  Corrected implementation:
  {text(row.get("corrected_code"))}
  """.strip()

          messages = [
              {
                  "role": "user",
                  "content": [
                      {
                          "type": "text",
                          "text": user_prompt,
                      }
                  ],
              },
              {
                  "role": "assistant",
                  "content": [
                      {
                          "type": "text",
                          "text": assistant_answer,
                      }
                  ],
              },
          ]

          fingerprint = hashlib.sha256(
              json.dumps(messages, sort_keys=True).encode()
          ).hexdigest()

          if fingerprint not in seen:
              seen.add(fingerprint)
              rows.append({
                  "messages": messages,
                  "has_image": False,
              })

  print("Usable records:", len(rows))

  assert len(rows) > 0, "No training records were loaded."

  split_index = max(1, int(len(rows) * 0.1))

  eval_rows = rows[:split_index]
  train_rows = rows[split_index:]

  train_ds = Dataset.from_list(train_rows)
  eval_ds = Dataset.from_list(eval_rows)

  print("Training records:", len(train_ds))
  print("Evaluation records:", len(eval_ds))

Usable records: 348
Training records: 314
Evaluation records: 34


In [21]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
      model=model,
      tokenizer=tokenizer,
      train_dataset=train_ds,
      eval_dataset=eval_ds,
      data_collator=UnslothVisionDataCollator(model, tokenizer),
      args=SFTConfig(
          output_dir=OUTPUT_DIR,
          max_seq_length=MAX_SEQ_LENGTH,
          per_device_train_batch_size=1,
          per_device_eval_batch_size=1,
          gradient_accumulation_steps=8,
          num_train_epochs=1,
          max_steps=10,  # Change to 1200 after testing
          learning_rate=2e-4,
          warmup_steps=50,
          weight_decay=0.01,
          lr_scheduler_type="cosine",
          optim="adamw_8bit",
          logging_steps=1,
          eval_strategy="steps",
          eval_steps=5,
          save_strategy="steps",
          save_steps=5,
          save_total_limit=2,
          fp16=True,
          bf16=False,
          report_to="none",
          remove_unused_columns=False,
          seed=SEED,
      ),
  )

print("Trainer created successfully.")

Unsloth: Model does not have a default image size - using 512
Trainer created successfully.


In [22]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        optim='adamw_8bit',
        logging_steps=5,
        eval_strategy='steps',
        eval_steps=25,
        save_strategy='steps',
        save_steps=25,
        save_total_limit=2,
        bf16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
        fp16=not (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8),
        report_to='none',
        remove_unused_columns=False,
        seed=SEED,
    ),
)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512
Trainable parameters: 39321600


In [23]:
# Train. For a quick smoke test, temporarily set max_steps=10 in SFTConfig above.
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved adapter to', OUTPUT_DIR)
print(train_result.metrics)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 314 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 39,321,600 of 4,477,137,408 (0.88% trained)


Step,Training Loss,Validation Loss
10,1.126355,1.053105


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/forma_v1_lora_run1/checkpoint-10/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/forma_v1_lora_run1/tokenizer_config.json.


Saved adapter to /kaggle/working/forma_v1_lora_run1
{'train_runtime': 541.3893, 'train_samples_per_second': 0.148, 'train_steps_per_second': 0.018, 'total_flos': 4613317909291008.0, 'train_loss': 1.2228396892547608, 'epoch': 0.25477707006369427}


In [26]:
# Save the exact normalized split used for reproducibility.
train_ds.to_json('/kaggle/working/uiux_train_normalized.jsonl')
eval_ds.to_json('/kaggle/working/uiux_eval_normalized.jsonl')

# Basic inference smoke test (text-only prompt works even when the dataset has images).
import torch
from transformers import TextStreamer

FastVisionModel.for_inference(model)

messages = [
    {
        'role': 'user',
        'content': [
            {
                'type': 'text',
                'text': 'Review a dashboard UI for hierarchy, accessibility, responsive behavior, and interaction quality. Return concise, actionable recommendations.'
            }
        ]
    }
]

# 1. Format the conversation using apply_chat_template
prompt_text = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

# 2. Explicitly pass 'text' as a keyword argument to prevent it from being parsed as an image
inputs = tokenizer(
    text=[prompt_text],
    images=None,
    return_tensors='pt'
).to('cuda')

# 3. Stream model outputs
streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs, 
    max_new_tokens=512, 
    temperature=0.7, 
    top_p=0.9, 
    streamer=streamer
)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✅ Hierarchy: Clear visual weight and grouping.  
⚠️ Accessibility: Contrast ratios below WCAG AA for text-on-dark.  
📱 Responsive: Breakpoints inconsistent; mobile menus collapse too early.  
🎯 Interaction: Hover states missing on key CTAs; micro-interactions absent.

🔧 Actionable Fixes:
- Boost contrast for all text (≥ 4.5:1 for normal text).
- Add hover states + subtle animations to primary buttons.
- Reorder nav hierarchy or expand collapsed menus with clear indicators.
- Define breakpoints (e.g., 768px, 1024px) and align grid system.

✅ Done: UI is clean.  
⚠️ Done: Accessibility + interactions need polish.  
🔧 Done: Responsive behavior needs refinement.

— 2-minute sprint fix: Revisit contrast + hover states + menu behavior.<|im_end|>


In [28]:
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/forma_v1_lora_run1")

for file in OUTPUT_DIR.rglob("*"):
      if file.is_file():
          print(file.relative_to(OUTPUT_DIR))

tokenizer.json
adapter_model.safetensors
training_args.bin
README.md
chat_template.jinja
processor_config.json
tokenizer_config.json
adapter_config.json
checkpoint-10/tokenizer.json
checkpoint-10/adapter_model.safetensors
checkpoint-10/scheduler.pt
checkpoint-10/training_args.bin
checkpoint-10/optimizer.pt
checkpoint-10/README.md
checkpoint-10/chat_template.jinja
checkpoint-10/trainer_state.json
checkpoint-10/scaler.pt
checkpoint-10/rng_state.pth
checkpoint-10/processor_config.json
checkpoint-10/tokenizer_config.json
checkpoint-10/adapter_config.json


In [48]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
login(token="Your_hf_token")

In [51]:
from huggingface_hub import create_repo

MODEL_REPO = "EgoisticCoder/forma-v1-lora"

create_repo(
      repo_id=MODEL_REPO,
      repo_type="model",
      private=False,
      exist_ok=True,
      token="Your_hf_token",
  )

RepoUrl('https://huggingface.co/EgoisticCoder/forma-v1-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='EgoisticCoder/forma-v1-lora')

In [52]:
from huggingface_hub import upload_folder

upload_folder(
      folder_path=str(OUTPUT_DIR),
      repo_id=MODEL_REPO,
      repo_type="model",
      token=HF_TOKEN,
      commit_message="Upload Forma V1 LoRA adapter",
      ignore_patterns=[
          "checkpoint-*",
          "optimizer.pt",
          "scheduler.pt",
          "rng_state.pth",
          "trainer_state.json",
          "*.bin",
      ],
  )

CommitInfo(commit_url='https://huggingface.co/EgoisticCoder/forma-v1-lora/commit/c5a33fe0ab56ff3e463e265c804cf3d2d199deb8', commit_message='Upload Forma V1 LoRA adapter', commit_description='', oid='c5a33fe0ab56ff3e463e265c804cf3d2d199deb8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/EgoisticCoder/forma-v1-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='EgoisticCoder/forma-v1-lora'), pr_revision=None, pr_num=None)

In [53]:

from huggingface_hub import list_repo_files

print(
      "\n".join(
          list_repo_files(
              repo_id=MODEL_REPO,
              repo_type="model",
              token="Your_hf_token",
          )
      )
  )

.gitattributes
README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
processor_config.json
tokenizer.json
tokenizer_config.json


In [54]:
from huggingface_hub import create_repo, upload_file

DATA_REPO = "EgoisticCoder/forma-v1-dataset"

create_repo(
      repo_id=DATA_REPO,
      repo_type="dataset",
      private=False,
      exist_ok=True,
      token="Your_hf_token",
  )

RepoUrl('https://huggingface.co/datasets/EgoisticCoder/forma-v1-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='EgoisticCoder/forma-v1-dataset')

In [55]:
upload_file(
      path_or_fileobj="/kaggle/working/Project_Infra/"
                     "ui-ux-model/dataset_gen_v2/v2_training_candidate.jsonl",
      path_in_repo="v2_training_candidate.jsonl",
      repo_id=DATA_REPO,
      repo_type="dataset",
      token="Your_hf_token",
      commit_message="Upload Forma V1 training candidate",
  )

CommitInfo(commit_url='https://huggingface.co/datasets/EgoisticCoder/forma-v1-dataset/commit/2bcccd43eb92b75789ec5f441de2e912ac9b5c09', commit_message='Upload Forma V1 training candidate', commit_description='', oid='2bcccd43eb92b75789ec5f441de2e912ac9b5c09', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/EgoisticCoder/forma-v1-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='EgoisticCoder/forma-v1-dataset'), pr_revision=None, pr_num=None)

In [64]:

upload_file(
      path_or_fileobj="/kaggle/working/Project_Infra/"
                     "ui-ux-model/dataset_gen_v2/v2_training_rejected.jsonl",
      path_in_repo="v2_training_rejected.jsonl",
      repo_id=DATA_REPO,
      repo_type="dataset",
      token="Your_hf_token",
  )

upload_file(
      path_or_fileobj="/kaggle/working/Project_Infra/"
                     "ui-ux-model/dataset_gen_v2/v2_training_candidate.manifest.json",
      path_in_repo="v2_training_candidate.manifest.json",
      repo_id=DATA_REPO,
      repo_type="dataset",
      token="Your_hf_token",
)

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/EgoisticCoder/forma-v1-dataset/commit/ab4e565aa69c41e5a2f85fa2d8b6f22bdf2be80f', commit_message='Upload v2_training_candidate.manifest.json with huggingface_hub', commit_description='', oid='ab4e565aa69c41e5a2f85fa2d8b6f22bdf2be80f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/EgoisticCoder/forma-v1-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='EgoisticCoder/forma-v1-dataset'), pr_revision=None, pr_num=None)